# CSCIE89 HW05 – Problem 4: Grad-CAM on the Problem 3 VGG16 flower model

**Goal:** use **Grad-CAM** (class-discriminative heat map for a single target convolutional layer) from Lab notebook **02_lab5_TL-resnet_xception** to see which parts of a flower image the **Problem 3 stage 2 model** used for its prediction.

**How this notebook is organised:**

1. **Sections 1–8 – rebuild the Problem 3 stage 2 model.** Problem 3 did not save the trained model, so these sections re-run the Problem 3 code unchanged (same data split, seed and settings) to recreate it.
2. **Section 9 – Grad-CAM tools**, copied from the lab notebook: `denormalize`, the `GradCAM` class and `show_gradcam`.
3. **Section 10 – apply Grad-CAM** to flower test images.
4. **Section 11 – report** on what the heat maps show.

**Target layer:** `conv_base[28]`, the **last convolutional layer of VGG16** (block 5, conv 3). It is the VGG16 counterpart of the lab's `resnet_model.layer4` and `xception_model.conv4`: the final, most class-specific feature maps before pooling and the classifier. It is also one of the layers fine-tuned in stage 2.

Use a GPU in Colab (**Runtime → Change runtime type → GPU**).

## 1. Import tools and choose training settings

Same as Problem 3, section 1.

In [ ]:
# Run once if packages are missing, then restart the kernel:
# %pip install torch torchvision scikit-learn matplotlib pillow

from pathlib import Path                                             # Tool for building folder/file paths.
import copy                                                          # Lets us save an independent copy of the model's weights.
import torch                                                         # PyTorch: the deep-learning library.
from torch import nn                                                 # Building blocks for networks (layers, loss functions).
from torch.utils.data import DataLoader, Subset                      # DataLoader feeds images in batches; Subset picks some images by their index.
from torchvision import datasets, models, transforms                 # Image-folder reader, pretrained models (VGG16), image resizing/augmenting.
from torchvision.datasets.utils import download_and_extract_archive  # Downloads a zip/tgz file and unpacks it.
from sklearn.model_selection import train_test_split                 # Splits data into train/validation/test.
from sklearn.metrics import classification_report, precision_score   # Precision/recall table, and overall precision.
import matplotlib.pyplot as plt                                      # For showing images.
from PIL import Image                                                # For opening image files.

SEED = 42                                        # Fixed random seed, so we get the same data split.
IMAGE_SIZE = 150                                 # Every photo is resized to 150 x 150 pixels.
BATCH_SIZE = 20                                  # The model looks at 20 photos at a time.
INITIAL_EPOCHS = 5                               # Stage 1: up to 5 passes over the training photos.
FINETUNE_EPOCHS = 3                              # Stage 2: up to 3 more passes.
INITIAL_LR = 2e-5                                # Stage 1 learning rate (size of each weight update).
FINETUNE_LR = 1e-5                               # Stage 2 learning rate: smaller, so VGG16's weights change gently.
PATIENCE = 3                                     # Stop a stage early if validation loss doesn't improve for 3 epochs.

torch.manual_seed(SEED)                                                        # Make PyTorch's random choices repeatable.
torch.set_num_threads(4)                                                       # Limit CPU threads (only matters when there is no GPU).
device = torch.device('cuda' if torch.cuda.is_available() else                 # Use the NVIDIA GPU if there is one (Colab),
                      'mps' if torch.backends.mps.is_available() else 'cpu')   # else an Apple GPU, else the CPU.
print('Using device:', device)                                                 # Show which one we are using.

## 2. Google Drive + local Colab setup: load the flower images

Same idea as Problem 3, section 2, with two changes:

- **Drive keeps the single archive file** `flower_photos.tgz` (229 MB) instead of 3,670 separate photos. Google Drive is very slow with thousands of small files; one large file copies in seconds and is then unpacked on Colab's fast disk.
- **Copies are checked for completeness:** the photo folder must hold all 3,670 photos and the archive must have its exact size. An interrupted copy (in an earlier run only `roses` was copied, which made every accuracy 100%) is replaced automatically.

In [ ]:
# GOOGLE DRIVE + LOCAL COLAB SETUP

from google.colab import drive                   # Colab tool to connect your Google Drive.
import shutil                                    # For copying files and folders.
from torchvision.datasets.utils import extract_archive  # Unpacks a .tgz archive.

drive.mount('/content/drive')                    # Connect Google Drive (asks for permission the first time).

# Permanent location in Google Drive
PROJECT_DIR = Path('/content/drive/MyDrive/CSCI-E-89-1-Deep-Learning/HW05')  # homework folder in Drive.
DRIVE_DATA_DIR = PROJECT_DIR / 'data'                                       # Data folder in Drive.
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)                           # Create it if it doesn't exist yet.

# IMPORTANT: # DATA_DIR is the fast temporary Colab drive

DATA_DIR = Path('/content/data')                                            # Colab's own disk: fast, but erased when the session ends.
DATA_DIR.mkdir(parents=True, exist_ok=True)                                 # Create it if it doesn't exist yet.


flower_root = DATA_DIR / 'flower_photos'                                    # Where training reads the flowers from (fast disk).
local_archive = DATA_DIR / 'flower_photos.tgz'                              # The downloaded archive (one 229 MB file).
drive_archive = DRIVE_DATA_DIR / 'flower_photos.tgz'                        # Permanent copy of the ARCHIVE in Drive (one file = fast to copy).
ARCHIVE_SIZE = 228_813_984                                                  # Exact size of a complete archive, in bytes.

def is_complete(folder):                                                    # The photo folder is complete only if it has all 3,670 photos
    return folder.exists() and sum(1 for _ in folder.rglob('*.jpg')) == 3670  # (an interrupted copy may hold only some classes).

def archive_ok(path):                                                       # An archive is complete only if it has the exact size.
    return path.exists() and path.stat().st_size == ARCHIVE_SIZE

# If flowers are not already (completely) in Colab
if not is_complete(flower_root):
    shutil.rmtree(flower_root, ignore_errors=True)                          # Remove any partial local copy first.

    # Copy the archive from Google Drive if a complete one is saved there, then unpack it
    if archive_ok(drive_archive):
        shutil.copy(drive_archive, local_archive)                           # Drive -> Colab disk: one file, a few seconds.
        extract_archive(str(local_archive), str(DATA_DIR))                  # Unpack the 3,670 photos on the fast disk.

    # Otherwise download them into Colab
    else:
        download_and_extract_archive(            # Download the flower archive and unpack it:
            'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz',  # from this web address,
            download_root=str(DATA_DIR),         # into the Colab disk.
        )

# Save a permanent copy of the archive to Google Drive if a complete one does not exist
if not archive_ok(drive_archive) and archive_ok(local_archive):
    shutil.copy(local_archive, drive_archive)                               # Colab disk -> Drive: one file, a few seconds.

print('Flower directory:', flower_root)          # Show where the photos are.
print('Photos found:', sum(1 for _ in flower_root.rglob('*.jpg')))  # Must be 3670.

## 3. Prepare images for VGG16

Same as Problem 3, section 3.

In [ ]:
MEAN = [0.485, 0.456, 0.406]                     # Average red, green, blue values of ImageNet photos.
STD = [0.229, 0.224, 0.225]                      # Spread of red, green, blue values of ImageNet photos.

train_transform = transforms.Compose([           # Steps applied to each TRAINING photo, in order:
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), # 1. Resize to 150 x 150.
    transforms.RandomHorizontalFlip(),           # 2. Mirror left-right, at random (half the time).
    transforms.RandomRotation(10),               # 3. Turn by a random angle between -10 and +10 degrees.
    transforms.ToTensor(),                       # 4. Turn the picture into numbers between 0 and 1.
    transforms.Normalize(MEAN, STD),             # 5. Rescale those numbers the same way VGG16 saw ImageNet photos.
])

eval_transform = transforms.Compose([            # Steps applied to VALIDATION and TEST photos (no random changes):
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), # 1. Resize to 150 x 150.
    transforms.ToTensor(),                       # 2. Turn the picture into numbers between 0 and 1.
    transforms.Normalize(MEAN, STD),             # 3. Rescale like ImageNet.
])

# ImageFolder assigns integer labels using alphabetically sorted folder names.
train_view = datasets.ImageFolder(flower_root, transform=train_transform)           # All 3,670 flowers, read with the TRAINING steps.
eval_view = datasets.ImageFolder(flower_root, transform=eval_transform)             # The same 3,670 flowers, read with the TEST steps.
class_names = train_view.classes                                                    # The 5 class names (folder names), in label order.
num_classes = len(class_names)                                                      # 5.
assert train_view.class_to_idx == eval_view.class_to_idx                            # Safety check: both views give each class the same number.
print('Class mapping:', train_view.class_to_idx)                                    # daisy=0, dandelion=1, roses=2, sunflowers=3, tulips=4
assert num_classes == 5 and len(eval_view) == 3670, 'Flower dataset incomplete: re-run section 2.'  # Stop if any class or photo is missing.

## 4. Separate training, validation, and test data

Same as Problem 3, section 4 (2,569 / 550 / 551 images).

In [ ]:
all_indices = list(range(len(eval_view)))                                          # Number every photo: 0, 1, 2, ..., 3669.
train_indices, remaining = train_test_split(                                       # Split 1: put 70% of the photo numbers in training, keep 30% aside.
    all_indices, test_size=0.30,                                                   # All photo numbers; 30% go to the 'aside' part.
    random_state=SEED, stratify=eval_view.targets,                                 # Same split every run; each part keeps the same mix of flower types.
)
val_indices, test_indices = train_test_split(                                      # Split 2: cut the 30% in half -> 15% validation, 15% test.
    remaining, test_size=0.50,                                                     # The 30% set aside; half of it goes to test.
    random_state=SEED, stratify=[eval_view.targets[i] for i in remaining],         # Again keep the same mix of flower types.
)

train_data = Subset(train_view, train_indices)                                     # Training photos (with random flips/rotations).
val_data = Subset(eval_view, val_indices)                                          # Validation photos (no random changes).
test_data = Subset(eval_view, test_indices)                                        # Test photos (no random changes).
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)            # Training photos in batches of 20, new random order each epoch.
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, num_workers=0)                              # Validation photos in batches of 20.
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, num_workers=0)                            # Test photos in batches of 20 (all 551 of them).

print('Training:', len(train_data), '| Validation:', len(val_data), '| Test:', len(test_data))       # 2569 | 550 | 551
images, labels = next(iter(train_loader))                                                            # Grab one batch to check its shape.
print('One batch:', images.shape, '| Labels:', labels.shape)                                         # [20 photos, 3 colors, 150, 150] and 20 labels.

## 5. Reuse VGG16 features and add our classifier

Same as Problem 3, section 5.

In [ ]:
pretrained = models.vgg16(weights=models.VGG16_Weights.DEFAULT)  # Download VGG16, already trained on ImageNet.
conv_base = pretrained.features                  # Keep only its convolution layers (the part that recognises patterns).
del pretrained  # Release the large original classifier that we do not need.

# Freeze all pretrained convolution weights for stage 1.
for parameter in conv_base.parameters():         # For every weight in VGG16's convolution layers:
    parameter.requires_grad = False              # freeze it (it is used, but training will not change it).

with torch.no_grad():                            # Quick test run, no training:
    feature_shape = conv_base(torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)).shape  # pass one blank image to see VGG16's output size.
    # Count the values from one image after flattening.
    number_of_features = feature_shape[1] * feature_shape[2] * feature_shape[3]  # 512 x 4 x 4 = 8,192 numbers per image.

model = nn.Sequential(                           # The full model, applied in this order:
    conv_base,                                   # 1. VGG16 turns the photo into 512 x 4 x 4 pattern scores.
    nn.Flatten(),                                # 2. Line them up into one list of 8,192 numbers.
    nn.Linear(number_of_features, 256),          # 3. New layer: 8,192 numbers -> 256 numbers.
    nn.ReLU(),                                   # 4. Set negative values to 0.
    nn.Dropout(0.5),                             # 5. During training, randomly switch off half of the 256 (reduces overfitting).
    nn.Linear(256, num_classes),                 # 6. New layer: 256 -> 5 scores, one per flower class (highest score = prediction).
).to(device)                                     # Put the model on the GPU (or CPU).
loss_function = nn.CrossEntropyLoss()            # Measures how wrong the 5 scores are vs. the true class (used for 3+ classes).
print('Convolution output:', tuple(feature_shape))  # (1, 512, 4, 4)
print('Flattened features:', number_of_features)    # 8192
print('Total parameters:', sum(p.numel() for p in model.parameters()))                          # All weights in the model.
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))   # Weights that will learn (only the 2 new layers).

## 6. Define one pass through the data, and the training loop

Same as Problem 3, sections 6 and 7.

In [ ]:
def run_epoch(loader, optimizer=None):                              # Go through all photos in `loader` once. With an optimizer it trains; without, it only measures.
    training = optimizer is not None                                # True = training pass, False = validation/test pass.
    model.train(training)                                           # Training mode turns Dropout on; evaluation mode turns it off.
    total_loss, correct, count = 0.0, 0, 0                          # Running totals, start at zero.
    # Gradients are needed for training but not for validation or testing.
    with torch.set_grad_enabled(training):
        for images, labels in loader:                               # Take one batch of 20 photos at a time:
            images = images.to(device)                              # put the photos on the GPU,
            labels = labels.to(device)                              # and their true classes (numbers 0-4).
            if training:                                            # Only when training:
                optimizer.zero_grad()                               # clear the previous batch's weight adjustments.
            logits = model(images)                                  # Model gives 5 scores per photo.
            loss = loss_function(logits, labels)                    # How wrong were the scores? (one number for the batch)
            if training:                                            # Only when training:
                loss.backward()                                     # work out how each weight should change,
                optimizer.step()                                    # and change the weights a little.
            total_loss += loss.item() * len(labels)                 # Add up the loss (times batch size, so the last smaller batch counts correctly).
            correct += (logits.argmax(1) == labels).sum().item()    # Count photos where the highest score is the true class.
            count += len(labels)                                    # Count photos seen.
    return total_loss / count, correct / count                      # Average loss and accuracy over all photos.

In [ ]:
def fit(epochs, learning_rate, stage_name):                                               # Train for up to `epochs` epochs and keep the best version of the model.
    # Create a new optimizer after changing which parameters are frozen.
    optimizer = torch.optim.RMSprop(                                                      # The optimizer is what updates the weights.
        [p for p in model.parameters() if p.requires_grad], lr=learning_rate,             # Give it only the unfrozen weights.
        alpha=0.9, eps=1e-7,                                                              # Technical settings of RMSprop.
    )
    history = {'train_loss': [], 'val_loss': [], 'train_accuracy': [], 'val_accuracy': []}  # Lists to record each epoch's results.

    # Evaluate the incoming model before changing any weights.
    best_loss, _ = run_epoch(val_loader)                                                  # Validation loss BEFORE this stage's training (the score to beat).
    best_weights = copy.deepcopy(model.state_dict())                                      # Save the starting weights as the best so far.
    best_epoch, waiting = 0, 0                                                            # Best epoch so far (0 = starting weights); epochs since last improvement.

    for epoch in range(1, epochs + 1):                                                    # For each epoch:
        train_loss, train_accuracy = run_epoch(train_loader, optimizer)                   # 1. Train on all training photos.
        val_loss, val_accuracy = run_epoch(val_loader)                                    # 2. Measure on the validation photos (no learning).
        history['train_loss'].append(train_loss)                                          # 3. Record this epoch's training loss,
        history['val_loss'].append(val_loss)                                              #    validation loss,
        history['train_accuracy'].append(train_accuracy)                                  #    training accuracy,
        history['val_accuracy'].append(val_accuracy)                                      #    and validation accuracy.

        # Select weights using validation loss, never test results.
        if val_loss < best_loss:                                                          # 4. If validation loss is the best so far:
            best_loss = val_loss                                                          #    remember it,
            best_weights = copy.deepcopy(model.state_dict())                              #    save a copy of these weights,
            best_epoch, waiting = epoch, 0                                                #    and reset the waiting counter.
        else:                                                                             #    Otherwise:
            waiting += 1                                                                  #    one more epoch without improvement.

        print(f'{stage_name} {epoch}/{epochs} | Train: {train_accuracy:.2%}'              # 5. Show progress: training accuracy,
              f' | Validation: {val_accuracy:.2%} | Val loss: {val_loss:.4f}', flush=True) #    validation accuracy and validation loss.
        if waiting >= PATIENCE:                                                           # 6. No improvement for 3 epochs in a row:
            print('Early stopping: validation loss stopped improving.')                    #    say so,
            break                                                                          #    stop this stage early.

    # Undo any later epochs that performed worse on validation data.
    model.load_state_dict(best_weights)                                                     # Go back to the best weights found.
    print(f'{stage_name}: restored best weights from epoch {best_epoch}.')                  # Say which epoch's weights are kept.
    return history                                                                          # Give back the recorded results.

## 7. Stage 1 — train only the new classifier

Same as Problem 3, section 8.

In [ ]:
history_initial = fit(INITIAL_EPOCHS, INITIAL_LR, 'Frozen base')  # Stage 1: VGG16 frozen, only the 2 new layers learn.
initial_val_loss, initial_val_accuracy = run_epoch(val_loader)    # Validation score of the best stage-1 model.
print(f'Selected stage-1 validation accuracy: {initial_val_accuracy:.2%}') # Show it.

## 8. Stage 2 — fine-tune block 5 (this is the model we inspect)

Same as Problem 3, section 9. **Added:** a quick test-set check, to confirm the recreated model matches Problem 3 (90.38% test accuracy there; small differences can occur because GPU training is not perfectly repeatable).

In [ ]:
# Module index 24 starts block 5 (it is not the 24th convolution).
for parameter in conv_base[24:].parameters():                             # For every weight in VGG16's last block (block 5):
    parameter.requires_grad = True                                        # unfreeze it so it can learn. Blocks 1-4 stay frozen.

print('Trainable convolution modules:')                                   # Heading for the list below.
for index, layer in enumerate(conv_base):                                 # Go through VGG16's layers
    if isinstance(layer, nn.Conv2d) and layer.weight.requires_grad:       # and list the convolution layers that can now learn
        print(f'  features[{index}]: {layer}')                            # (should be the 3 in block 5).
print('Total trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))  # Now block 5 + the 2 new layers.

history_finetune = fit(FINETUNE_EPOCHS, FINETUNE_LR, 'Fine-tuning')  # Stage 2: continue training, starting from the stage-1 model.
final_val_loss, final_val_accuracy = run_epoch(val_loader)           # Validation score of the best stage-2 model.
print(f'Selected final validation accuracy: {final_val_accuracy:.2%}') # Show it.
test_loss, test_accuracy = run_epoch(test_loader)                    # Check: test accuracy of the recreated stage-2 model
print(f'Stage 2 test accuracy: {test_accuracy:.2%} (Problem 3: 90.38%)') # should be close to Problem 3's result.

## 9. Grad-CAM tools (from the lab notebook)

The three pieces below are the lab notebook's code, adapted to this model (`DEVICE` → `device`, `CLASS_NAMES` → `class_names`, cats/dogs → 5 flower classes):

- **`denormalize`** undoes the ImageNet normalization so the image can be displayed with its real colors.
- **`GradCAM`** attaches two **hooks** to the target layer:
  - a *forward hook* saves the layer's output (**activations**, one 2-D map per channel);
  - a *backward hook* saves the **gradients** of the chosen class score with respect to those activations.

  Then: average each channel's gradients → one **weight per channel** (how much that channel supports the class) → weighted sum of the activation maps → keep only positive evidence (ReLU) → resize to the image size → scale to 0–1.
- **`show_gradcam`** plots the input image, the heat map, and the heat map laid over the image.

**One change needed for VGG16:** VGG16's ReLU layers work *in place* (`inplace=True`), overwriting the conv layer's output. PyTorch's backward hook does not allow that, so we switch them to `inplace=False`. This changes memory use only; the model's outputs stay exactly the same.

In [ ]:
# ============================================================
# Undo ImageNet normalization for plotting
# ============================================================

def denormalize(image):                                     # Normalized image tensor -> displayable image.
    """
    Reverse ImageNet normalization for plotting.
    """
    mean = torch.tensor(MEAN, dtype=image.dtype).view(3, 1, 1)  # Channel means as a 3x1x1 tensor.
    std = torch.tensor(STD, dtype=image.dtype).view(3, 1, 1)    # Channel standard deviations as 3x1x1.
    return (image.detach().cpu() * std + mean).clamp(0, 1)      # x * std + mean, kept within 0..1.


# ============================================================
# Grad-CAM: class-discriminative heat-map visualization
# ============================================================

class GradCAM:
    """
    Grad-CAM for a single target convolutional layer.

    The heat map emphasizes locations whose activation channels,
    weighted by gradients, positively support a selected class logit.
    """

    def __init__(self, model, target_layer):               # Attach hooks to the target layer.
        self.model = model                                  # The trained network.
        self.activations = None                             # Will hold the layer's output maps.
        self.gradients = None                               # Will hold the gradients of those maps.

        self.forward_handle = (
            target_layer.register_forward_hook(self._save_activations)       # Save output on the forward pass.
        )

        self.backward_handle = (
            target_layer.register_full_backward_hook(self._save_gradients)   # Save gradients on the backward pass.
        )

    def _save_activations(self, module, inputs, output):    # Called automatically during the forward pass.
        self.activations = output                           # Layer output: [1, channels, height, width].

    def _save_gradients(self, module, grad_input, grad_output):  # Called automatically during the backward pass.
        self.gradients = grad_output[0]                     # Gradient of the class score w.r.t. that output.

    def generate(self, image, target_class=None):           # Build the heat map for one image.
        """
        Return:
            cam: normalized 2D heat map
            predicted_class: model prediction
            probabilities: flower-class softmax probabilities

        If target_class is None, use the predicted class.
        """

        self.model.eval()                                   # Evaluation mode (Dropout off).

        image_batch = image.unsqueeze(0).to(device)         # Add batch dimension: [1, 3, 150, 150].

        self.model.zero_grad(set_to_none=True)              # Clear old gradients.

        logits = self.model(image_batch)                    # Forward pass: 5 class scores (hook saves activations).

        predicted_class = int(logits.argmax(dim=1).item())  # Class with the highest score.

        if target_class is None:                            # No class chosen:
            target_class = predicted_class                  # explain the predicted class.

        # Differentiate the selected class score with respect
        # to the target layer's output activations.
        logits[0, target_class].backward()                  # Backward pass (hook saves gradients).

        # Spatially average gradients to get one weight per channel.
        channel_weights = self.gradients.mean(dim=(2, 3), keepdim=True)   # [1, 512, 1, 1]: importance of each channel.

        # Weighted activation sum, retaining only positive evidence.
        cam = torch.relu((channel_weights * self.activations).sum(dim=1, keepdim=True))  # [1, 1, 9, 9] heat map.

        # Resize to the original displayed image dimensions.
        cam = torch.nn.functional.interpolate(cam, size=image.shape[-2:], mode="bilinear", align_corners=False)  # -> 150 x 150.

        cam = cam.squeeze().detach().cpu()                  # Drop extra dimensions: [150, 150].

        # Normalize to [0, 1] for visualization.
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)          # Strongest evidence = 1.

        probabilities = torch.softmax(logits, dim=1).detach().cpu().squeeze(0)  # 5 class probabilities.

        return (cam, predicted_class, probabilities)        # Heat map, prediction, probabilities.

    def close(self):
        """
        Remove hooks so they do not accumulate after reruns.
        """
        self.forward_handle.remove()                        # Detach the forward hook.
        self.backward_handle.remove()                       # Detach the backward hook.


# ============================================================
# Plot input image, Grad-CAM heat map, and overlay
# ============================================================

def show_gradcam(model, target_layer, image, true_label, model_name):  # Compute and plot Grad-CAM for one image.
    """
    Generate and display Grad-CAM for the model's predicted class.
    """

    cam_helper = GradCAM(model=model, target_layer=target_layer)       # Attach the hooks.

    try:
        cam, prediction, probabilities = (
            cam_helper.generate(image=image, target_class=None)        # Heat map for the predicted class.
        )

    finally:
        cam_helper.close()                                  # Always remove the hooks afterwards.

    image_np = denormalize(image).permute(1, 2, 0).numpy()  # [3, H, W] -> [H, W, 3] for plotting.

    confidence = float(probabilities[prediction].item())   # Probability of the predicted class.

    title_color = ("green" if int(true_label) == prediction else "crimson")  # Green = correct, red = wrong.

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))      # Three panels side by side.

    # Original image.
    axes[0].imshow(image_np)                                # Panel 1: the flower.
    axes[0].set_title(("Input Image\n" f"True: {class_names[int(true_label)]}"), fontsize=11)  # True class.
    axes[0].axis("off")                                     # Hide axes.

    # Grad-CAM heat map only.
    axes[1].imshow(cam.numpy(), cmap="jet", vmin=0, vmax=1) # Panel 2: heat map (red = strong evidence).
    axes[1].set_title("Grad-CAM Heat Map", fontsize=11)     # Title.
    axes[1].axis("off")                                     # Hide axes.

    # Overlay.
    axes[2].imshow(image_np)                                # Panel 3: the flower...
    axes[2].imshow(cam.numpy(), cmap="jet", alpha=0.45, vmin=0, vmax=1)  # ...with the heat map on top.
    axes[2].set_title(("Overlay\n" f"Pred: {class_names[prediction]} " f"({confidence:.1%})"),
                      color=title_color, fontsize=11)       # Prediction and confidence.
    axes[2].axis("off")                                     # Hide axes.

    fig.suptitle(f"{model_name}: Grad-CAM", fontweight="bold")  # Figure title.
    fig.subplots_adjust(top=0.80, wspace=0.12)              # Spacing.
    plt.show()                                              # Display.

## 10. Grad-CAM on flower images from the Problem 3 stage 2 model

As in the lab, we take a test image (`CAM_INDEX = 0`, the first test image) and show Grad-CAM for the **predicted class** at the single target layer `conv_base[28]` (VGG16's last convolutional layer, 9 × 9 feature maps at our 150 × 150 input).

The lab compared two models on one image; with one model we instead show **two images**:

1. The first test image (`CAM_INDEX = 0`).
2. The **sunflower** test image that the Problem 3 model **misclassified as daisy** (section 11 of Problem 3), to see where the model looked when it was wrong.

Change `CAM_INDEX` to inspect any other test image.

In [ ]:
# ============================================================
# Grad-CAM on the Problem 3 stage 2 model
# ============================================================

for layer in conv_base:                                     # VGG16's ReLUs overwrite their input in place,
    if isinstance(layer, nn.ReLU):                          # which the backward hook does not allow,
        layer.inplace = False                               # so make them return a new tensor (same results).

target_layer = conv_base[28]                                # Single target layer: last conv layer (block 5, conv 3).
print('Target layer:', target_layer)                        # Show it.

CAM_INDEX = 0                                               # Which test image to inspect.

image, true_label = test_data[CAM_INDEX]                    # Normalized image tensor and its true class.

print("Grad-CAM true class:", class_names[int(true_label)]) # Show the true class.

# 1. First test image.
show_gradcam(
    model=model,                                            # Problem 3 stage 2 model.
    target_layer=target_layer,                              # conv_base[28].
    image=image,                                            # The flower.
    true_label=true_label,                                  # Its true class.
    model_name=f"VGG16 stage 2, test image {CAM_INDEX}",    # Figure title.
)

# 2. The sunflower that Problem 3 misclassified (first sunflower in the test set).
sunflower_class = class_names.index('sunflowers')           # Label number of sunflowers (3).
sunflower_index = next(i for i, index in enumerate(test_indices)
                       if eval_view.targets[index] == sunflower_class)  # Position of the first sunflower in test_data.
image, true_label = test_data[sunflower_index]              # That image and its true class.

show_gradcam(
    model=model,                                            # Problem 3 stage 2 model.
    target_layer=target_layer,                              # conv_base[28].
    image=image,                                            # The sunflower.
    true_label=true_label,                                  # Its true class.
    model_name=f"VGG16 stage 2, sunflower (test image {sunflower_index})",  # Figure title.
)

## 11. Report on results

_Fill in after running the notebook._

**What Grad-CAM shows:** bright (red/yellow) regions are where the target layer's feature maps most strongly supported the predicted class; blue regions contributed little. Bright regions indicate relatively stronger model evidence, not proof that the model "understands" the flower.

| Image | True class | Predicted (confidence) | Where the heat map is strongest |
|:---|:---|:---|:---|
| Test image 0 | | | |
| Sunflower | sunflowers | | |

Points to discuss:

1. For the correct prediction, does the heat map sit on the flower itself (petals, center) or on the background?
2. For the misclassified sunflower, what did the model focus on? Does that explain why it chose daisy?